In [0]:
# ============================================================
# NOTEBOOK: silver_produtos_categorias
# Squad 1 · Dupla 3 — Bruna Diniz & Rafael Miranda
# ============================================================

In [0]:
%pip install deltalake --quiet
dbutils.library.restartPython()

In [0]:

import os
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql import Window
from deltalake import DeltaTable, write_deltalake

load_dotenv("/Workspace/Repos/diniz.dbruna@gmail.com/estagio-empregadados-turma-2/.env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

bronze_produtos_path = f"{lakehouse_base}/bronze/ecommerce_produtos"
bronze_categorias_path = f"{lakehouse_base}/bronze/ecommerce_categorias"
silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}

def ler_delta_direto(caminho):
    dt = DeltaTable(caminho, storage_options=storage_options)
    return spark.createDataFrame(dt.to_pandas())

print("Setup concluído.")

In [0]:
def processar_silver(caminho_bronze, caminho_silver, aplicar_regras):
    df_bronze = ler_delta_direto(caminho_bronze)

    try:
        dt_silver = DeltaTable(caminho_silver, storage_options=storage_options)
        ja_processados = set(dt_silver.to_pandas(columns=["bronze_source_file"])["bronze_source_file"].unique())
        df_micro_lote = df_bronze.filter(~F.col("bronze_source_file").isin(ja_processados))
    except Exception:
        df_micro_lote = df_bronze

    df_silver = aplicar_regras(df_micro_lote).withColumn(
        "silver_processed_at", F.current_timestamp()
    )

    df_pandas = df_silver.toPandas()
    df_pandas["bronze_ingested_at"] = df_pandas["bronze_ingested_at"].dt.tz_localize("UTC")
    df_pandas["silver_processed_at"] = df_pandas["silver_processed_at"].dt.tz_localize("UTC")

    # Força colunas de regra a serem booleanas de verdade, mesmo quando
    # totalmente nulas (pendentes) -> evita erro de tipo "Null" no delta-rs
    for coluna in df_pandas.columns:
        if coluna.startswith("regra_"):
            df_pandas[coluna] = df_pandas[coluna].astype("boolean")

    write_deltalake(caminho_silver, df_pandas, mode="append", storage_options=storage_options)

    print(f"OK [{caminho_silver}] {len(df_pandas)} linhas processadas")
    return df_silver

In [0]:
def regras_categorias(df):
    # Tabela de referência DEDUPLICADA por (arquivo, id) -> evita fan-out em joins
    df_ref = df.select("bronze_source_file", "id_categoria", "id_categoria_pai") \
                .dropDuplicates(["bronze_source_file", "id_categoria"])

    # 1) Técnica: id_categoria não pode ser nulo nem duplicado DENTRO DO MESMO ARQUIVO
    w_dup = Window.partitionBy("bronze_source_file", "id_categoria")
    df = df.withColumn("_id_count", F.count("id_categoria").over(w_dup))
    df = (df
        .withColumn("regra_01_id_valido",
            F.col("id_categoria").isNotNull() & (F.col("_id_count") == 1))
        .withColumn("regra_02_nome_preenchido",
            F.col("nome_categoria").isNotNull() & (F.trim(F.col("nome_categoria")) != ""))
        .withColumn("regra_04_sem_ciclo_direto",
            F.col("id_categoria_pai").isNull() | (F.col("id_categoria_pai") != F.col("id_categoria")))
        # ------------------------------------------------------------------
        # DECISÃO DE NEGÓCIO (confirmado com Geovany, 28/09/2026):
        # O caractere '&' foi removido da lista de caracteres inválidos da
        # regra 9. Motivo: várias categorias legítimas do catálogo usam '&'
        # no nome (ex.: "Adega & Destilados", "Bazar & Utilidades"), e o
        # caractere sozinho não representa risco de XSS — só precisa ser
        # escapado (&amp;) na exibição em HTML, responsabilidade do
        # front-end. A regra passa a sinalizar apenas padrões que indicam
        # corrupção/injeção real de dado: '<', '>' e a palavra 'script'.
        # ------------------------------------------------------------------
        .withColumn("regra_09_nome_sem_caracteres_invalidos",
            ~F.col("nome_categoria").rlike(r'[<>]|script'))
    ).drop("_id_count")

    # 5) Técnica: nome único dentro do mesmo nível, DENTRO DO MESMO ARQUIVO
    w_nivel = Window.partitionBy("bronze_source_file", F.coalesce(F.col("id_categoria_pai"), F.lit(-1)), "nome_categoria")
    df = df.withColumn("_nome_nivel_count", F.count("nome_categoria").over(w_nivel))
    df = df.withColumn("regra_05_nome_unico_no_nivel", F.col("_nome_nivel_count") == 1).drop("_nome_nivel_count")

    # 3) Técnica: id_categoria_pai deve existir em id_categoria, NO MESMO ARQUIVO
    df = (df
        .join(df_ref.select(
                F.col("bronze_source_file").alias("_bsf"),
                F.col("id_categoria").alias("_id_ref")),
              (df["bronze_source_file"] == F.col("_bsf")) & (df["id_categoria_pai"] == F.col("_id_ref")),
              "left")
        .withColumn("regra_03_pai_existe", F.col("id_categoria_pai").isNull() | F.col("_id_ref").isNotNull())
        .drop("_bsf", "_id_ref")
    )

    # 6) Negócio: hierarquia deve ter exatamente 2 níveis, NO MESMO ARQUIVO
    df = (df
        .join(df_ref.select(
                F.col("bronze_source_file").alias("_bsf2"),
                F.col("id_categoria").alias("_id_pai"),
                F.col("id_categoria_pai").alias("_pai_do_pai")),
              (df["bronze_source_file"] == F.col("_bsf2")) & (df["id_categoria_pai"] == F.col("_id_pai")),
              "left")
        .withColumn("regra_06_dois_niveis",
            F.col("id_categoria_pai").isNull() | F.col("_pai_do_pai").isNull())
        .drop("_bsf2", "_id_pai", "_pai_do_pai")
    )

    # 8) Negócio: toda categoria raiz deve ter >=1 subcategoria, NO MESMO ARQUIVO
    subcats_por_arquivo = (df_ref
        .filter(F.col("id_categoria_pai").isNotNull())
        .groupBy("bronze_source_file", F.col("id_categoria_pai").alias("_pai_com_filho"))
        .count()
    )
    df = (df
        .join(subcats_por_arquivo,
              (df["bronze_source_file"] == subcats_por_arquivo["bronze_source_file"]) &
              (df["id_categoria"] == subcats_por_arquivo["_pai_com_filho"]),
              "left")
        .withColumn("regra_08_raiz_tem_subcategoria",
            F.when(F.col("id_categoria_pai").isNotNull(), F.lit(True))
             .otherwise(F.col("count").isNotNull()))
        .drop(subcats_por_arquivo["bronze_source_file"], "_pai_com_filho", "count")
    )

    # Dependem de outras tabelas / dados externos -> ficam pendentes por enquanto
    df = df.withColumn("regra_07_subcategoria_tem_produto", F.lit(None).cast("boolean"))
    df = df.withColumn("regra_10_total_raizes_confere", F.lit(None).cast("boolean"))

    return df

In [0]:
processar_silver(bronze_categorias_path, silver_categorias_path, regras_categorias)

In [0]:
display(
    spark.read.format("delta").table("silver.ecommerce_categorias")
    .filter(F.col("nome_categoria").contains("&"))
    .select("nome_categoria", "regra_09_nome_sem_caracteres_invalidos")
    .distinct()
)

In [0]:
def regras_produtos(df, df_categorias_silver):
    # Referência de categorias: pega uma linha por id_categoria (dado estável, sem precisar escopar por arquivo)
    df_cat_ref = (df_categorias_silver
        .select("id_categoria", "id_categoria_pai")
        .dropDuplicates(["id_categoria"])
    )

    # 1) Técnica: sku não pode ser nulo nem duplicado DENTRO DO MESMO ARQUIVO
    w_dup = Window.partitionBy("bronze_source_file", "sku")
    df = df.withColumn("_sku_count", F.count("sku").over(w_dup))

    df = (df
        .withColumn("regra_01_sku_valido",
            F.col("sku").isNotNull() & (F.col("_sku_count") == 1))

        # 2) Técnica: preco_lista deve ser > 0
        .withColumn("regra_02_preco_positivo",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") > 0))

        # 3) Técnica: nome_produto não pode ser nulo ou vazio
        .withColumn("regra_03_nome_preenchido",
            F.col("nome_produto").isNotNull() & (F.trim(F.col("nome_produto")) != ""))

        # 5) Técnica: unidade_medida na lista permitida
        .withColumn("regra_05_unidade_valida",
            F.col("unidade_medida").isin("un", "kg", "g", "L", "ml"))

        # 6) Negócio: is_ativo deve ser booleano e não nulo
        .withColumn("regra_06_is_ativo_valido",
            F.col("is_ativo").isNotNull())

        # 7) Negócio: preco_lista <= 5000
        .withColumn("regra_07_preco_dentro_limite",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") <= 5000))

        # 8) Negócio: nome_marca não pode ser nulo ou vazio
        .withColumn("regra_08_marca_preenchida",
            F.col("nome_marca").isNotNull() & (F.trim(F.col("nome_marca")) != ""))
         ).drop("_sku_count")

        # 4) Técnica: id_categoria deve existir em ecommerce_categorias (FK)
        # 10) Negócio: SKU deve pertencer a subcategoria (id_categoria_pai IS NOT NULL)
    df = (df
        .join(df_cat_ref.select(
                F.col("id_categoria").alias("_cat_id"),
                F.col("id_categoria_pai").alias("_cat_pai")),
              df["id_categoria"] == F.col("_cat_id"), "left")
        .withColumn("regra_04_categoria_existe", F.col("_cat_id").isNotNull())
        .withColumn("regra_10_pertence_subcategoria",
            F.col("_cat_id").isNotNull() & F.col("_cat_pai").isNotNull())
        .drop("_cat_id", "_cat_pai")
        )

        # 9) Negócio: produto ativo deve ter >=1 venda nos últimos 90 dias
        # PENDENTE: depende da Silver de ecommerce_itens_pedido (Squad 1, Dupla 1 —
        # Gabriel Franz Simoni & Carlos Eduardo Santos de Souza). Assim que a tabela
        # estiver disponível, plugar via join (sku) + filtro de dt_pedido >= hoje-90d.
    df = df.withColumn("regra_09_ativo_com_venda_90d", F.lit(None).cast("boolean"))

    return df

In [0]:
df_categorias_silver = ler_delta_direto(silver_categorias_path)

processar_silver(bronze_produtos_path, silver_produtos_path,
                  lambda df: regras_produtos(df, df_categorias_silver))

In [0]:
from pyspark.sql import functions as F

df_prod = ler_delta_direto(silver_produtos_path)
colunas_regra = [c for c in df_prod.columns if c.startswith("regra_")]

df_prod.select([
    F.sum(F.when(F.col(c) == False, 1).otherwise(0)).alias(c) for c in colunas_regra
]).show(truncate=False, vertical=True)

In [0]:
from pyspark.sql import functions as F

df_cat = ler_delta_direto(silver_categorias_path)
colunas_regra_cat = [c for c in df_cat.columns if c.startswith("regra_")]

df_cat.select([
    F.sum(F.when(F.col(c) == False, 1).otherwise(0)).alias(c) for c in colunas_regra_cat
]).show(truncate=False, vertical=True)

In [0]:
processar_silver(bronze_categorias_path, silver_categorias_path, regras_categorias)
processar_silver(bronze_produtos_path, silver_produtos_path,
                  lambda df: regras_produtos(df, df_categorias_silver))

In [0]:
dt_silver_prod = DeltaTable(silver_produtos_path, storage_options=storage_options)
dt_silver_cat = DeltaTable(silver_categorias_path, storage_options=storage_options)
print("Silver produtos:", len(dt_silver_prod.to_pandas()))
print("Silver categorias:", len(dt_silver_cat.to_pandas()))

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
datalake_service = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential,
)

In [0]:
squad_fs_client = datalake_service.get_file_system_client("squad1")

paths_no_squad1 = [p.name for p in squad_fs_client.get_paths(recursive=True) if p.is_directory or True]
itens_pedido_encontrados = [p for p in paths_no_squad1 if "itens_pedido" in p.lower()]

print("Caminhos relacionados a itens_pedido encontrados:")
for p in itens_pedido_encontrados[:20]:
    print(" -", p)

if not itens_pedido_encontrados:
    print("Nada encontrado ainda.")